In [1]:
# ============================================================
# Day 3 — XGBoost Model
# Goal: Beat Random Forest v2 (81.87%) and reach 85%+
# ============================================================

import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

# XGBoost — our new powerful model
from xgboost import XGBClassifier

# Metrics to evaluate our model
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    confusion_matrix, 
    roc_auc_score
)

print("✅ All libraries imported successfully")
print(f"XGBoost version: {__import__('xgboost').__version__}")

✅ All libraries imported successfully
XGBoost version: 3.2.0


In [2]:
# ============================================================
# Load preprocessed data from Day 2
# These files were saved in models/ folder
# ============================================================

X_train = joblib.load('../models/X_train.pkl')
X_test  = joblib.load('../models/X_test.pkl')
y_train = joblib.load('../models/y_train.pkl')
y_test  = joblib.load('../models/y_test.pkl')

print("✅ Data loaded successfully\n")
print(f"X_train: {X_train.shape}  →  {X_train.shape[0]:,} training samples, {X_train.shape[1]} features")
print(f"X_test:  {X_test.shape}   →  {X_test.shape[0]:,} test samples, {X_test.shape[1]} features")
print(f"y_train: {y_train.shape}  →  {y_train.shape[0]:,} training labels")
print(f"y_test:  {y_test.shape}   →  {y_test.shape[0]:,} test labels")

# Quick sanity check on label distribution
print(f"\n📊 Class distribution:")
print(f"   Training:  {(y_train == 0).sum():,} normal | {(y_train == 1).sum():,} attacks")
print(f"   Testing:   {(y_test == 0).sum():,} normal | {(y_test == 1).sum():,} attacks")
print(f"\n   Attack ratio in training: {y_train.mean():.2%}")
print(f"   Attack ratio in testing:  {y_test.mean():.2%}")

✅ Data loaded successfully

X_train: (125973, 21)  →  125,973 training samples, 21 features
X_test:  (22544, 21)   →  22,544 test samples, 21 features
y_train: (125973,)  →  125,973 training labels
y_test:  (22544,)   →  22,544 test labels

📊 Class distribution:
   Training:  67,343 normal | 58,630 attacks
   Testing:   9,711 normal | 12,833 attacks

   Attack ratio in training: 46.54%
   Attack ratio in testing:  56.92%


In [3]:
# ============================================================
# Calculate scale_pos_weight for handling class imbalance
# This is XGBoost's equivalent of class_weight='balanced'
# ============================================================

# Count negative class (normal = 0) and positive class (attack = 1)
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()

# Formula: negatives ÷ positives
scale_pos_weight = neg_count / pos_count

print(f"Normal samples (negatives): {neg_count:,}")
print(f"Attack samples (positives): {pos_count:,}")
print(f"\nscale_pos_weight = {neg_count:,} / {pos_count:,} = {scale_pos_weight:.4f}")
print(f"\nThis tells XGBoost: 'each attack sample is worth {scale_pos_weight:.2f}× a normal sample'")

Normal samples (negatives): 67,343
Attack samples (positives): 58,630

scale_pos_weight = 67,343 / 58,630 = 1.1486

This tells XGBoost: 'each attack sample is worth 1.15× a normal sample'


In [4]:
# ============================================================
# Train Baseline XGBoost Model
# Goal: See where XGBoost starts before any tuning
# ============================================================

import time

# Create the XGBoost classifier with our chosen hyperparameters
xgb_baseline = XGBClassifier(
    n_estimators=200,           # Build 200 trees
    max_depth=6,                # Each tree up to 6 levels deep
    learning_rate=0.1,          # Each tree contributes 10% of its prediction
    scale_pos_weight=scale_pos_weight,  # Handle class imbalance
    eval_metric='logloss',      # How XGBoost measures its own progress
    random_state=42,            # For reproducibility
    n_jobs=-1,                  # Use all CPU cores (faster training)
    verbosity=0                 # Don't print training progress
)

print("🏗️  Training XGBoost (this takes ~30 seconds)...\n")
start = time.time()

# .fit() trains the model on our data
xgb_baseline.fit(X_train, y_train)

elapsed = time.time() - start
print(f"✅ Training complete in {elapsed:.1f} seconds")
print(f"   Built {xgb_baseline.n_estimators} trees, each up to depth {xgb_baseline.max_depth}")

🏗️  Training XGBoost (this takes ~30 seconds)...

✅ Training complete in 1.1 seconds
   Built 200 trees, each up to depth 6


In [5]:
# ============================================================
# Evaluate Baseline XGBoost
# How does it perform on the test set?
# ============================================================

# Get predictions
y_pred_baseline = xgb_baseline.predict(X_test)

# Get predicted probabilities (needed for ROC-AUC)
y_proba_baseline = xgb_baseline.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy   = accuracy_score(y_test, y_pred_baseline)
roc_auc    = roc_auc_score(y_test, y_proba_baseline)

print("=" * 60)
print("📊 BASELINE XGBOOST RESULTS")
print("=" * 60)
print(f"Accuracy:  {accuracy:.4f}  ({accuracy:.2%})")
print(f"ROC-AUC:   {roc_auc:.4f}")
print()
print(f"Comparison with Day 2 Random Forest v2:")
print(f"   RF v2 (with threshold tuning):  81.87%")
print(f"   XGBoost baseline (no tuning):   {accuracy:.2%}")
print(f"   Difference: {(accuracy - 0.8187)*100:+.2f}%")
print("=" * 60)

📊 BASELINE XGBOOST RESULTS
Accuracy:  0.8013  (80.13%)
ROC-AUC:   0.9608

Comparison with Day 2 Random Forest v2:
   RF v2 (with threshold tuning):  81.87%
   XGBoost baseline (no tuning):   80.13%
   Difference: -1.74%


In [6]:
# ============================================================
# Threshold Tuning for Baseline XGBoost
# Find the threshold that gives us the best accuracy
# (Same approach we used on Day 2's RF)
# ============================================================

# Try many thresholds between 0.20 and 0.60
thresholds = np.arange(0.20, 0.60, 0.01)

# Store results in a list for later plotting
results = []

# Loop through each threshold
for thresh in thresholds:
    # Apply this threshold to convert probabilities into 0/1 predictions
    y_pred_thresh = (y_proba_baseline >= thresh).astype(int)
    
    # Compute accuracy at this threshold
    acc = accuracy_score(y_test, y_pred_thresh)
    
    # Save the result
    results.append({'threshold': thresh, 'accuracy': acc})

# Convert results to a DataFrame for easy analysis
results_df = pd.DataFrame(results)

# Find the best threshold
best_row = results_df.loc[results_df['accuracy'].idxmax()]
best_threshold = best_row['threshold']
best_accuracy  = best_row['accuracy']

print("=" * 60)
print("🎯 THRESHOLD TUNING RESULTS")
print("=" * 60)
print(f"Default threshold (0.50) accuracy: {accuracy:.4f}  ({accuracy:.2%})")
print(f"Best threshold:                    {best_threshold:.2f}")
print(f"Best accuracy:                     {best_accuracy:.4f}  ({best_accuracy:.2%})")
print(f"Improvement from threshold tuning: +{(best_accuracy - accuracy)*100:.2f}%")
print("=" * 60)

# Show top 5 thresholds for context
print("\n📊 Top 5 thresholds by accuracy:")
print(results_df.nlargest(5, 'accuracy').to_string(index=False))

print("\n📊 Comparison vs Day 2's Random Forest v2:")
print(f"   RF v2 (threshold 0.32):              81.87%")
print(f"   XGBoost (threshold {best_threshold:.2f}):      {best_accuracy:.2%}")
print(f"   Difference: {(best_accuracy - 0.8187)*100:+.2f}%")

🎯 THRESHOLD TUNING RESULTS
Default threshold (0.50) accuracy: 0.8013  (80.13%)
Best threshold:                    0.24
Best accuracy:                     0.8210  (82.10%)
Improvement from threshold tuning: +1.97%

📊 Top 5 thresholds by accuracy:
 threshold  accuracy
      0.24  0.820972
      0.25  0.820396
      0.26  0.819730
      0.27  0.819553
      0.28  0.818178

📊 Comparison vs Day 2's Random Forest v2:
   RF v2 (threshold 0.32):              81.87%
   XGBoost (threshold 0.24):      82.10%
   Difference: +0.23%


In [7]:
# ============================================================
# Train TUNED XGBoost
# Goal: Push past RF v2 by a meaningful margin (target 85%+)
# ============================================================

import time

xgb_tuned = XGBClassifier(
    n_estimators=500,             # More trees — early stopping will pick optimal count
    max_depth=8,                  # Slightly deeper trees
    learning_rate=0.05,           # Smaller, more careful steps
    subsample=0.8,                # Each tree sees 80% of rows
    colsample_bytree=0.8,         # Each tree sees 80% of features
    min_child_weight=3,           # Leaves need at least 3 samples
    gamma=0.1,                    # Small barrier against weak splits
    reg_alpha=0.1,                # L1 regularization
    reg_lambda=1.0,               # L2 regularization
    scale_pos_weight=scale_pos_weight,  # Same class imbalance handling
    eval_metric='logloss',        
    early_stopping_rounds=20,     # Stop if no improvement for 20 rounds
    random_state=42,              
    n_jobs=-1,                    
    verbosity=0
)

print("🏗️  Training TUNED XGBoost...")
print(f"   Max trees: 500, but early stopping will cut us off if we plateau\n")
start = time.time()

# Notice the new argument: eval_set
# This is what XGBoost monitors to decide when to early-stop
xgb_tuned.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],  # Watch performance on this set
    verbose=False                  # Don't print every tree's progress
)

elapsed = time.time() - start
print(f"✅ Training complete in {elapsed:.1f} seconds")
print(f"   Actual trees used: {xgb_tuned.best_iteration + 1}  (early stopping kicked in if this is below 500)")
print(f"   Best iteration's logloss: {xgb_tuned.best_score:.4f}")

🏗️  Training TUNED XGBoost...
   Max trees: 500, but early stopping will cut us off if we plateau

✅ Training complete in 0.4 seconds
   Actual trees used: 27  (early stopping kicked in if this is below 500)
   Best iteration's logloss: 0.4550


In [8]:
# ============================================================
# DIAGNOSTIC: Check the 27-tree model's actual performance
# Don't trust logloss — check accuracy directly
# ============================================================

# Get probabilities from the early-stopped model
y_proba_quick = xgb_tuned.predict_proba(X_test)[:, 1]

# Check accuracy at default threshold
acc_quick = accuracy_score(y_test, (y_proba_quick >= 0.5).astype(int))
auc_quick = roc_auc_score(y_test, y_proba_quick)

print(f"27-tree model — default threshold: {acc_quick:.2%}")
print(f"27-tree model — ROC-AUC:           {auc_quick:.4f}")

# Quick threshold sweep
best_acc, best_t = 0, 0
for t in np.arange(0.20, 0.60, 0.01):
    a = accuracy_score(y_test, (y_proba_quick >= t).astype(int))
    if a > best_acc:
        best_acc, best_t = a, t

print(f"27-tree model — best threshold:    {best_t:.2f}")
print(f"27-tree model — best accuracy:     {best_acc:.2%}")

27-tree model — default threshold: 78.17%
27-tree model — ROC-AUC:           0.9443
27-tree model — best threshold:    0.20
27-tree model — best accuracy:     87.77%


In [9]:
# ============================================================
# DIAGNOSTIC: Search a WIDER threshold range
# Check if accuracy peaks below 0.20 (our previous lower bound)
# ============================================================

# Extended search: from 0.05 to 0.60
extended_thresholds = np.arange(0.05, 0.60, 0.01)

# Reuse the 27-tree model's probabilities
results_extended = []
for thresh in extended_thresholds:
    y_pred = (y_proba_quick >= thresh).astype(int)
    acc = accuracy_score(y_test, y_pred)
    results_extended.append({'threshold': thresh, 'accuracy': acc})

results_ext_df = pd.DataFrame(results_extended)

# Find the true best
best_row = results_ext_df.loc[results_ext_df['accuracy'].idxmax()]
print(f"True best threshold: {best_row['threshold']:.2f}")
print(f"True best accuracy:  {best_row['accuracy']:.2%}")

# Show top 10 in case there are ties
print("\n📊 Top 10 thresholds (wider search):")
print(results_ext_df.nlargest(10, 'accuracy').to_string(index=False))

# Also show the bottom range to confirm we've explored enough
print("\n📉 Lowest range (0.05 - 0.15) for reference:")
print(results_ext_df.head(11).to_string(index=False))

True best threshold: 0.16
True best accuracy:  89.42%

📊 Top 10 thresholds (wider search):
 threshold  accuracy
      0.16  0.894163
      0.15  0.892433
      0.17  0.890703
      0.18  0.889549
      0.14  0.888795
      0.19  0.886089
      0.20  0.877706
      0.21  0.871629
      0.13  0.870431
      0.22  0.867193

📉 Lowest range (0.05 - 0.15) for reference:
 threshold  accuracy
      0.05  0.569242
      0.06  0.569242
      0.07  0.569242
      0.08  0.569242
      0.09  0.569242
      0.10  0.569242
      0.11  0.569242
      0.12  0.569242
      0.13  0.870431
      0.14  0.888795
      0.15  0.892433


In [10]:
# ============================================================
# Train XGBoost v2 (PROPERLY TUNED)
# Fixing the early-stopping-too-soon problem
# ============================================================

import time

xgb_v2 = XGBClassifier(
    n_estimators=500,             # Up to 500 trees
    max_depth=8,                  # Same as before
    learning_rate=0.08,           # Slightly higher than 0.05 for faster learning
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric=['logloss', 'error'],  # Watch BOTH metrics
    early_stopping_rounds=50,     # More patience (was 20)
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

print("🏗️  Training XGBoost v2 (with proper patience)...")
print(f"   Max trees: 500, patience: 50 rounds\n")
start = time.time()

xgb_v2.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

elapsed = time.time() - start
print(f"✅ Training complete in {elapsed:.1f} seconds")
print(f"   Trees actually used: {xgb_v2.best_iteration + 1}")
print(f"   Best iteration's metric: {xgb_v2.best_score:.4f}")

🏗️  Training XGBoost v2 (with proper patience)...
   Max trees: 500, patience: 50 rounds

✅ Training complete in 0.4 seconds
   Trees actually used: 1
   Best iteration's metric: 0.1796


In [11]:
# ============================================================
# Train XGBoost v3 — NO early stopping
# Just train all 300 trees and trust the regularization
# ============================================================

import time

xgb_v3 = XGBClassifier(
    n_estimators=300,             # Fixed: train all 300 trees
    max_depth=8,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',        # Single metric, no early stopping
    random_state=42,
    n_jobs=-1,
    verbosity=0
    # NOTE: NO early_stopping_rounds parameter
)

print("🏗️  Training XGBoost v3 (no early stopping)...")
print(f"   Will train ALL 300 trees, no early termination\n")
start = time.time()

# NOTE: No eval_set this time either — we don't want any monitoring drama
xgb_v3.fit(X_train, y_train)

elapsed = time.time() - start
print(f"✅ Training complete in {elapsed:.1f} seconds")
print(f"   Trees used: {xgb_v3.n_estimators} (all of them, as planned)")

🏗️  Training XGBoost v3 (no early stopping)...
   Will train ALL 300 trees, no early termination

✅ Training complete in 1.5 seconds
   Trees used: 300 (all of them, as planned)


In [12]:
# ============================================================
# Evaluate XGBoost v3 — the properly-trained model
# ============================================================

# Get predictions
y_pred_v3  = xgb_v3.predict(X_test)
y_proba_v3 = xgb_v3.predict_proba(X_test)[:, 1]

# Default threshold metrics
acc_v3_default = accuracy_score(y_test, y_pred_v3)
auc_v3         = roc_auc_score(y_test, y_proba_v3)

print(f"v3 — Default threshold (0.5): {acc_v3_default:.2%}")
print(f"v3 — ROC-AUC: {auc_v3:.4f}\n")

# Full threshold sweep (0.05 to 0.60)
thresholds_full = np.arange(0.05, 0.60, 0.01)
results_v3 = []
for thresh in thresholds_full:
    acc = accuracy_score(y_test, (y_proba_v3 >= thresh).astype(int))
    results_v3.append({'threshold': thresh, 'accuracy': acc})

results_v3_df = pd.DataFrame(results_v3)
best_row = results_v3_df.loc[results_v3_df['accuracy'].idxmax()]
best_thresh_v3 = best_row['threshold']
best_acc_v3    = best_row['accuracy']

print(f"🎯 Best threshold: {best_thresh_v3:.2f}")
print(f"🎯 Best accuracy:  {best_acc_v3:.2%}")

print("\n📊 Top 10 thresholds:")
print(results_v3_df.nlargest(10, 'accuracy').to_string(index=False))

# Comparison
print("\n" + "=" * 70)
print("🏆 FINAL MODEL COMPARISON")
print("=" * 70)
print(f"{'Model':<35} {'Accuracy':<11} {'Threshold':<11} {'ROC-AUC':<8}")
print("-" * 70)
print(f"{'RF v2 (Day 2)':<35} {'81.87%':<11} {'0.32':<11} {'n/a':<8}")
print(f"{'XGB baseline (200t)':<35} {'82.10%':<11} {'0.24':<11} {'0.9608':<8}")
print(f"{'XGB v3 (300t, no early stop)':<35} {f'{best_acc_v3:.2%}':<11} {f'{best_thresh_v3:.2f}':<11} {f'{auc_v3:.4f}':<8}")
print("=" * 70)

v3 — Default threshold (0.5): 80.55%
v3 — ROC-AUC: 0.9673

🎯 Best threshold: 0.06
🎯 Best accuracy:  87.27%

📊 Top 10 thresholds:
 threshold  accuracy
      0.06  0.872738
      0.07  0.870121
      0.05  0.868879
      0.08  0.867104
      0.09  0.864931
      0.10  0.863290
      0.11  0.862314
      0.12  0.861116
      0.13  0.859031
      0.14  0.857124

🏆 FINAL MODEL COMPARISON
Model                               Accuracy    Threshold   ROC-AUC 
----------------------------------------------------------------------
RF v2 (Day 2)                       81.87%      0.32        n/a     
XGB baseline (200t)                 82.10%      0.24        0.9608  
XGB v3 (300t, no early stop)        87.27%      0.06        0.9673  


In [13]:
# ============================================================
# Train XGBoost v4 — Remove scale_pos_weight
# Theory: our data is nearly balanced, so weighting hurts more than helps
# ============================================================

import time

xgb_v4 = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    # scale_pos_weight=scale_pos_weight,  ← REMOVED
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

print("🏗️  Training XGBoost v4 (no scale_pos_weight)...")
print(f"   Hypothesis: removing class weighting will give better-calibrated probabilities\n")
start = time.time()

xgb_v4.fit(X_train, y_train)

elapsed = time.time() - start
print(f"✅ Training complete in {elapsed:.1f} seconds")

# Quick evaluation
y_proba_v4 = xgb_v4.predict_proba(X_test)[:, 1]

# Default threshold
acc_v4_default = accuracy_score(y_test, (y_proba_v4 >= 0.5).astype(int))
auc_v4 = roc_auc_score(y_test, y_proba_v4)

print(f"\nv4 — Default threshold (0.5): {acc_v4_default:.2%}")
print(f"v4 — ROC-AUC: {auc_v4:.4f}")

# Full threshold sweep
thresholds_full = np.arange(0.05, 0.95, 0.01)
results_v4 = []
for thresh in thresholds_full:
    acc = accuracy_score(y_test, (y_proba_v4 >= thresh).astype(int))
    results_v4.append({'threshold': thresh, 'accuracy': acc})

results_v4_df = pd.DataFrame(results_v4)
best_row = results_v4_df.loc[results_v4_df['accuracy'].idxmax()]
best_thresh_v4 = best_row['threshold']
best_acc_v4    = best_row['accuracy']

print(f"\n🎯 Best threshold: {best_thresh_v4:.2f}")
print(f"🎯 Best accuracy:  {best_acc_v4:.2%}")

print("\n📊 Top 10 thresholds:")
print(results_v4_df.nlargest(10, 'accuracy').to_string(index=False))

🏗️  Training XGBoost v4 (no scale_pos_weight)...
   Hypothesis: removing class weighting will give better-calibrated probabilities

✅ Training complete in 1.6 seconds

v4 — Default threshold (0.5): 80.74%
v4 — ROC-AUC: 0.9669

🎯 Best threshold: 0.05
🎯 Best accuracy:  87.72%

📊 Top 10 thresholds:
 threshold  accuracy
      0.05  0.877218
      0.06  0.874867
      0.07  0.871629
      0.08  0.868923
      0.09  0.866749
      0.10  0.864487
      0.11  0.862669
      0.12  0.859519
      0.13  0.856237
      0.14  0.854374


In [14]:
# ============================================================
# Proper evaluation: carve a validation set out of training data
# Test set (KDDTest+) stays UNTOUCHED until the very end
# ============================================================

from sklearn.model_selection import train_test_split

# Split training data: 80% for training, 20% for validation
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.20,        # 20% goes to validation
    random_state=42,       # reproducible split
    stratify=y_train       # keep the same attack/normal ratio in both halves
)

print("Data split complete:")
print(f"   Training:   {X_tr.shape[0]:,} samples  ({y_tr.mean():.1%} attacks)")
print(f"   Validation: {X_val.shape[0]:,} samples  ({y_val.mean():.1%} attacks)")
print(f"   Test:       {X_test.shape[0]:,} samples  ({y_test.mean():.1%} attacks)  ← untouched")

Data split complete:
   Training:   100,778 samples  (46.5% attacks)
   Validation: 25,195 samples  (46.5% attacks)
   Test:       22,544 samples  (56.9% attacks)  ← untouched


In [15]:
# ============================================================
# Train XGBoost on X_tr (the 80% training portion)
# Same hyperparameters as v3 — a solid, honest configuration
# ============================================================

import time

xgb_final = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

print("🏗️  Training final XGBoost on 80% training split...")
start = time.time()
xgb_final.fit(X_tr, y_tr)
print(f"✅ Done in {time.time() - start:.1f}s")

🏗️  Training final XGBoost on 80% training split...
✅ Done in 1.4s


In [16]:
# ============================================================
# Choose threshold using VALIDATION set only
# The test set is NOT involved in this decision
# ============================================================

# Get probabilities on the VALIDATION set
y_proba_val = xgb_final.predict_proba(X_val)[:, 1]

# Sweep thresholds on validation
thresholds = np.arange(0.05, 0.95, 0.01)
val_results = []
for thresh in thresholds:
    acc = accuracy_score(y_val, (y_proba_val >= thresh).astype(int))
    val_results.append({'threshold': thresh, 'accuracy': acc})

val_df = pd.DataFrame(val_results)
best_val = val_df.loc[val_df['accuracy'].idxmax()]
chosen_threshold = best_val['threshold']

print("🎯 Threshold chosen on VALIDATION set:")
print(f"   Best validation threshold: {chosen_threshold:.2f}")
print(f"   Validation accuracy:       {best_val['accuracy']:.2%}")
print(f"\n   This threshold is now LOCKED. We do not change it after seeing test.")

print("\n📊 Validation top 5 thresholds:")
print(val_df.nlargest(5, 'accuracy').to_string(index=False))

🎯 Threshold chosen on VALIDATION set:
   Best validation threshold: 0.46
   Validation accuracy:       99.89%

   This threshold is now LOCKED. We do not change it after seeing test.

📊 Validation top 5 thresholds:
 threshold  accuracy
      0.46  0.998928
      0.47  0.998928
      0.48  0.998928
      0.43  0.998889
      0.44  0.998889


In [17]:
# ============================================================
# FINAL HONEST EVALUATION
# Apply the validation-chosen threshold to the test set ONCE
# Whatever number comes out — that's our real accuracy. No re-tuning.
# ============================================================

from sklearn.metrics import classification_report, confusion_matrix

# Probabilities on the test set
y_proba_test = xgb_final.predict_proba(X_test)[:, 1]

# Apply the LOCKED threshold from validation
y_pred_test = (y_proba_test >= chosen_threshold).astype(int)

# Honest metrics
honest_acc = accuracy_score(y_test, y_pred_test)
honest_auc = roc_auc_score(y_test, y_proba_test)

print("=" * 60)
print("🏁 HONEST TEST RESULT (threshold locked from validation)")
print("=" * 60)
print(f"Threshold used:    {chosen_threshold:.2f}  (chosen on validation, not test)")
print(f"Test accuracy:     {honest_acc:.2%}")
print(f"Test ROC-AUC:      {honest_auc:.4f}")
print("=" * 60)

print("\n📋 Classification report:")
print(classification_report(y_test, y_pred_test, target_names=['Normal', 'Attack']))

print("📊 Confusion matrix:")
cm = confusion_matrix(y_test, y_pred_test)
tn, fp, fn, tp = cm.ravel()
print(f"   ✅ Normal correctly ignored:  {tn:,}")
print(f"   ✅ Attacks correctly caught:  {tp:,}")
print(f"   ❌ False alarms:              {fp:,}")
print(f"   ❌ Attacks missed:            {fn:,}")

print("\n" + "=" * 60)
print("HONEST COMPARISON vs Day 2")
print("=" * 60)
print(f"   RF v2 (Day 2):                81.87%  (threshold likely test-tuned)")
print(f"   XGBoost (honest, validation): {honest_acc:.2%}")

🏁 HONEST TEST RESULT (threshold locked from validation)
Threshold used:    0.46  (chosen on validation, not test)
Test accuracy:     79.56%
Test ROC-AUC:      0.9658

📋 Classification report:
              precision    recall  f1-score   support

      Normal       0.69      0.97      0.80      9711
      Attack       0.97      0.66      0.79     12833

    accuracy                           0.80     22544
   macro avg       0.83      0.82      0.80     22544
weighted avg       0.85      0.80      0.79     22544

📊 Confusion matrix:
   ✅ Normal correctly ignored:  9,425
   ✅ Attacks correctly caught:  8,511
   ❌ False alarms:              286
   ❌ Attacks missed:            4,322

HONEST COMPARISON vs Day 2
   RF v2 (Day 2):                81.87%  (threshold likely test-tuned)
   XGBoost (honest, validation): 79.56%


In [18]:
# ============================================================
# Day 3 — Save the final XGBoost model + locked threshold
# ============================================================

import json

# Save the trained model
joblib.dump(xgb_final, '../models/xgb_model_v1.pkl')

# Save the threshold + honest metrics (chosen on validation, evaluated once on test)
xgb_metadata = {
    'model':              'XGBoost v1',
    'chosen_threshold':   float(chosen_threshold),   # 0.46, locked from validation
    'test_accuracy':      round(float(honest_acc), 4),
    'test_roc_auc':       round(float(honest_auc), 4),
    'threshold_chosen_on': 'validation set (no test leakage)',
    'notes':              'Honest evaluation. Test set used once. Accuracy reflects NSL-KDD distribution shift.'
}

with open('../models/xgb_metadata.json', 'w') as f:
    json.dump(xgb_metadata, f, indent=4)

print("✅ Saved xgb_model_v1.pkl")
print("✅ Saved xgb_metadata.json")
print("\n📄 Metadata contents:")
print(json.dumps(xgb_metadata, indent=4))

# Confirm what's in the models folder now
import os
print("\n📁 models/ folder now contains:")
for f in sorted(os.listdir('../models/')):
    print(f"   {f}")

✅ Saved xgb_model_v1.pkl
✅ Saved xgb_metadata.json

📄 Metadata contents:
{
    "model": "XGBoost v1",
    "chosen_threshold": 0.4600000000000001,
    "test_accuracy": 0.7956,
    "test_roc_auc": 0.9658,
    "threshold_chosen_on": "validation set (no test leakage)",
    "notes": "Honest evaluation. Test set used once. Accuracy reflects NSL-KDD distribution shift."
}

📁 models/ folder now contains:
   X_test.pkl
   X_train.pkl
   feature_importance.png
   feature_selector.pkl
   random_forest_model.pkl
   rf_model_v2.pkl
   scaler.pkl
   threshold.json
   xgb_metadata.json
   xgb_model_v1.pkl
   y_test.pkl
   y_train.pkl


In [1]:
# ============================================================
# Day 4 kickoff — verify Day 3 artifacts survived
# ============================================================

import os
import json
import joblib

print("📁 Checking models/ folder contents:\n")
for f in sorted(os.listdir('../models/')):
    print(f"   {f}")

print("\n🔍 Loading Day 3 XGBoost artifacts...")
xgb_model = joblib.load('../models/xgb_model_v1.pkl')
with open('../models/xgb_metadata.json') as f:
    xgb_meta = json.load(f)

print("✅ Model loaded successfully")
print(f"\n📄 Day 3 metadata:")
print(json.dumps(xgb_meta, indent=4))

print(f"\n🌳 Model check: {xgb_model.n_estimators} trees, max_depth={xgb_model.max_depth}")

📁 Checking models/ folder contents:

   X_test.pkl
   X_train.pkl
   feature_importance.png
   feature_selector.pkl
   random_forest_model.pkl
   rf_model_v2.pkl
   scaler.pkl
   threshold.json
   xgb_metadata.json
   xgb_model_v1.pkl
   y_test.pkl
   y_train.pkl

🔍 Loading Day 3 XGBoost artifacts...
✅ Model loaded successfully

📄 Day 3 metadata:
{
    "model": "XGBoost v1",
    "chosen_threshold": 0.4600000000000001,
    "test_accuracy": 0.7956,
    "test_roc_auc": 0.9658,
    "threshold_chosen_on": "validation set (no test leakage)",
    "notes": "Honest evaluation. Test set used once. Accuracy reflects NSL-KDD distribution shift."
}

🌳 Model check: 300 trees, max_depth=8
